# 2_train_gpu0
training our tuning fork networks, setting to use gpu0 to avoid memory conflict with other notebooks needing gpu allocation

In [1]:
#### misc
import pandas as pd
import numpy as np
import os
from pathlib import Path
import pickle
import time
from itertools import product

#### graphical
import matplotlib.pyplot as plt

#### ML
import sklearn
from sklearn.decomposition import PCA
import tensorflow as tf
import keras
from keras import layers

#### custom
from InversePCA import InversePCA
from WMSE import WMSE, WMSE_metric

##### poke gpu
os.environ["CUDA_VISIBLE_DEVICES"]="0"

physical_devices = tf.config.list_physical_devices("GPU") 

gpu0usage = tf.config.experimental.get_memory_info("GPU:0")["current"]

print("Current GPU usage:\n"
     + " - GPU0: " + str(gpu0usage) + "B\n")

2024-02-21 17:41:00.174603: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-02-21 17:41:00.174629: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-02-21 17:41:00.175465: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-02-21 17:41:00.180156: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-02-21 17:41:00.752407: W tensorflow/compiler/tf2

Current GPU usage:
 - GPU0: 0B



2024-02-21 17:41:01.307862: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 18042 MB memory:  -> device: 0, name: NVIDIA RTX A4500, pci bus id: 0000:41:00.0, compute capability: 8.6


## data prep
load in data and perform final prep (normalisation, label definition) before we start training)

In [2]:
#### load in grid
#df_full = pd.read_hdf('/home/oxs235/datastorage/repos_data/' + str(path) + '/data/df_all_log.h5', key='df')
df_full = pd.read_hdf('/home/oxs235/datastorage/repos_data/ojscutt/pitchfork/data/keystone/bob.h5', key='df')

#### define inputs
inputs = ['log_initial_mass_std', 'log_initial_Zinit_std', 'log_initial_Yinit_std', 'log_initial_MLT_std', 'log_star_age_std']

#### define outputs
classical_outputs = ['log_radius_std', 'log_luminosity_std', 'log_surface_Z_std']
astero_outputs = ['log_nu_max_std', 'log_delta_nu_std']

outputs = classical_outputs+astero_outputs

#### train/test split
seed = 42

df_train = df_full.sample(frac=0.9, random_state=seed)

df_train_inputs, df_val_inputs, df_train_outputs, df_val_outputs = sklearn.model_selection.train_test_split(df_train[inputs],df_train[outputs], test_size = 0.1, random_state=seed)

print("Training set: ", len(df_train_inputs))
print("Validation set: ", len(df_val_inputs))

#### can't have too many describes
df_full.describe()

Training set:  641680
Validation set:  71298


,initial_mass,initial_Zinit,initial_Yinit,initial_MLT,star_age,radius,luminosity,effective_T,surface_Z,nu_max,...,log_initial_Zinit_std,log_initial_Yinit_std,log_initial_MLT_std,log_star_age_std,log_radius_std,log_luminosity_std,log_effective_T_std,log_surface_Z_std,log_nu_max_std,log_delta_nu_std
count,792198.000000,792198.000000,792198.000000,792198.000000,792198.000000,792198.000000,792198.000000,792198.000000,792198.000000,792198.000000,...,7.921980e+05,7.921980e+05,7.921980e+05,7.921980e+05,7.921980e+05,7.921980e+05,7.921980e+05,7.921980e+05,7.921980e+05,7.921980e+05
mean,1.020928,0.014158,0.281968,1.800434,5.337817,1.392278,2.188875,5769.945558,0.013057,2144.494362,...,1.096169e-15,-7.220987e-15,-4.786453e-15,-4.070877e-16,-1.176767e-16,1.699136e-16,-1.712454e-14,-7.229759e-16,2.140693e-16,4.743244e-15
std,0.115940,0.009543,0.028031,0.099999,3.500584,0.496634,1.567627,571.439401,0.009448,1114.943956,...,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00
min,0.800000,0.003869,0.240000,1.700000,0.029664,0.716057,0.132426,4099.843268,0.001355,431.750387,...,-1.587889e+00,-1.552692e+00,-1.004351e+00,-5.401508e+00,-1.838429e+00,-3.357128e+00,-3.411526e+00,-2.567988e+00,-2.374067e+00,-2.378665e+00
25%,0.920000,0.006293,0.260000,1.700000,2.540991,1.026434,1.013962,5395.170704,0.005270,1201.566502,...,-8.681881e-01,-7.567156e-01,-1.004351e+00,-4.871917e-01,-7.492884e-01,-6.726386e-01,-6.308042e-01,-8.133896e-01,-6.873370e-01,-6.699680e-01
50%,1.020000,0.010769,0.280000,1.900000,4.663161,1.233171,1.834036,5753.825658,0.010201,2086.086045,...,-7.323067e-02,-1.975705e-02,9.956664e-01,1.832412e-01,-1.942661e-01,1.089368e-01,2.104667e-02,3.987237e-02,2.217758e-01,2.196204e-01
75%,1.120000,0.020073,0.300000,1.900000,7.687273,1.642987,2.934617,6112.460646,0.018846,2908.642565,...,8.480524e-01,6.663350e-01,9.956664e-01,7.352247e-01,6.735927e-01,7.288367e-01,6.334358e-01,8.329647e-01,7.695485e-01,7.748266e-01
max,1.200000,0.038980,0.320000,1.900000,13.999974,2.873431,11.612924,7995.935261,0.039035,5714.523485,...,1.830007e+00,1.308131e+00,9.956664e-01,1.397210e+00,2.364354e+00,2.542864e+00,3.353879e+00,1.773685e+00,1.882448e+00,1.718511e+00


## gridsearch parameters
define gridsearch parameters for the tuning/pitchfork setup, focus on hparams that alter overarching architecture

In [3]:
"""
DEFINE TARGET ARCHITECTURES FOR GRID SEARCH
Rerun after training to avoid "___ not iterable" errors
"""
stem_d_layers = [4]
stem_d_units = [64]

ctine_d_layers = [4]
ctine_d_units = [64]

atine_d_layers = [4]
atine_d_units = [64]

archs = pd.DataFrame(product(stem_d_layers, stem_d_units, ctine_d_layers, ctine_d_units, atine_d_layers, atine_d_units))

archs.columns = ['stem_d_layers', 'stem_d_units', 'ctine_d_layers', 'ctine_d_units', 'atine_d_layers', 'atine_d_units']

In [ ]:
"""
        ________
_______/
       \________
| stem | tines |

"""


tf.keras.backend.clear_session()


for arch_i in range(len(archs)):
    tf.keras.backend.clear_session()
    arch = archs.iloc[[arch_i]]
    
    ######## stem
    #### input
    stem_input = keras.Input(shape=(len(inputs),))

    #### dense layers
    stem_d_layers = arch["stem_d_layers"].iloc[0]
    stem_d_units = arch["stem_d_units"].iloc[0]

    for stem_d_layer in range(stem_d_layers):
        if stem_d_layer == 0:
            stem = layers.Dense(stem_d_units, activation='relu')(stem_input)
            stem = layers.LayerNormalization()(stem)
        else:
            stem = layers.Dense(stem_d_units, activation='relu')(stem)
            stem = layers.LayerNormalization()(stem)

    ######## classical tine
    #### dense layers
    ctine_d_layers = arch["ctine_d_layers"].iloc[0]
    ctine_d_units = arch["ctine_d_units"].iloc[0]

    for ctine_d_layer in range(ctine_d_layers):
        if ctine_d_layer == 0:
            ctine = layers.Dense(ctine_d_units, activation='relu')(stem)
            ctine = layers.LayerNormalization()(ctine)
        else:
            ctine = layers.Dense(ctine_d_units, activation='relu')(ctine)
            ctine = layers.LayerNormalization()(ctine)

    #### output
    ctine_out = layers.Dense(len(classical_outputs),name='classical_outs')(ctine)


    ######## astero tine
    #### dense layers
    atine_d_layers = arch["atine_d_layers"].iloc[0]
    atine_d_units = arch["atine_d_units"].iloc[0]
    
    for atine_d_layer in range(atine_d_layers):
        if atine_d_layer == 0:
            atine = layers.Dense(atine_d_units, activation='relu')(stem)
            atine = layers.LayerNormalization()(atine)
        else:
            atine = layers.Dense(atine_d_units, activation='relu')(atine)
            atine = layers.LayerNormalization()(atine)

    #### output
    atine_out = layers.Dense(int(len(astero_outputs)))(atine)

    ######## construct and fit
    model = keras.Model(inputs=stem_input, outputs=[ctine_out, atine_out], name='tuning_fork')

    #### compile model
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

        
    model.compile(loss='MSE', optimizer=optimizer)
    
    #### fit model
    arch_name = 'bob1'

    log_dir = "/home/oxs235/datastorage/repos_data/ojscutt/pitchfork/logs/keystone/" + arch_name

    def scheduler(epoch, lr):
        if lr < 5e-5:
            return lr
        else:
            return lr * tf.math.exp(-0.001)

    lr_callback = tf.keras.callbacks.LearningRateScheduler(scheduler, verbose=0)
                                                       
    cp_callback = tf.keras.callbacks.ModelCheckpoint("/home/oxs235/datastorage/repos_data/ojscutt/pitchfork/models/keystone/" + arch_name + ".h5",
                                                     monitor= 'val_loss',
                                                     save_best_only= True,
                                                     save_freq='epoch')    

    tb_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir) 

    history = model.fit(df_train_inputs,
                        [df_train_outputs[classical_outputs],df_train_outputs[astero_outputs]],
                        validation_data=(df_val_inputs,[df_val_outputs[classical_outputs], df_val_outputs[astero_outputs]]),
                        batch_size=2**12,
                        verbose=1,
                        epochs=10000,
                        callbacks=[lr_callback, cp_callback, tb_callback],
                        shuffle=True
                       )  

2024-02-21 17:41:04.312591: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


Epoch 1/10000
